# 🎯 UNITEE - Automated Public Market Surveillance

**Project**: Monitor French public markets for modular building opportunities

**Notebook Tabs**:
1. ✅ **Setup** - Configuration & Database Connection
2. ✅ **Extraction** - Fetch data from APIs (data.gouv.fr, BOAMP)
3. **Transformation** - Clean & normalize data
4. **Validation** - Quality checks & doublon detection
5. **Export** - SQL INSERT & CSV generation

---

## Tab 1: Setup & Configuration

In [ ]:
# 1.1 Import required libraries
import os
import sys
import json
import yaml
import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta
from typing import Dict, List, Tuple
import mysql.connector
from mysql.connector import Error as MySQLError
import logging
from pathlib import Path

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ All required libraries imported successfully")

In [ ]:
# 1.2 Load environment configuration
from dotenv import load_dotenv

# Load .env file
env_path = Path('../.env')
if env_path.exists():
    load_dotenv(env_path)
    print(f"✅ Environment variables loaded from {env_path}")
else:
    print(f"⚠️ .env file not found at {env_path}")

# Get database configuration from environment
DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'port': int(os.getenv('DB_PORT', 3306)),
    'user': os.getenv('DB_USER', 'unitee_user'),
    'password': os.getenv('DB_PASS', 'UniteeStrong1234'),
    'database': os.getenv('DB_NAME', 'unitee'),
    'charset': os.getenv('DB_CHARSET', 'utf8mb4'),
}

print("\n📋 Database Configuration:")
print(f"  Host: {DB_CONFIG['host']}:{DB_CONFIG['port']}")
print(f"  Database: {DB_CONFIG['database']}")
print(f"  User: {DB_CONFIG['user']}")
print(f"  Charset: {DB_CONFIG['charset']}")

In [ ]:
# 1.3 Load config.yaml
config_path = Path('../config/config.yaml')

if config_path.exists():
    with open(config_path, 'r', encoding='utf-8') as f:
        CONFIG = yaml.safe_load(f)
    print(f"✅ Configuration loaded from {config_path}")
    
    # Display key configuration
    print("\n📊 Configuration Summary:")
    print(f"  Project: {CONFIG.get('project', 'UNITEE')}")
    print(f"  Keywords (PRIMARY): {len(CONFIG.get('keywords', {}).get('primary', []))} items")
    print(f"  Keywords (SECONDARY): {len(CONFIG.get('keywords', {}).get('secondary', []))} items")
    if 'schedule' in CONFIG:
        print(f"  Extraction Schedule: {CONFIG['schedule'].get('frequency', 'daily')}")
else:
    print(f"⚠️ config.yaml not found at {config_path}")
    CONFIG = {'project': 'UNITEE', 'keywords': {'primary': [], 'secondary': []}}

In [ ]:
# 1.4 Test MySQL Connection
def test_database_connection(config: Dict) -> Tuple[bool, str]:
    """
    Test connection to MySQL database.
    Returns: (success: bool, message: str)
    """
    try:
        conn = mysql.connector.connect(**config)
        if conn.is_connected():
            cursor = conn.cursor()
            cursor.execute("SELECT DATABASE(), USER(), VERSION()")
            db, user, version = cursor.fetchone()
            cursor.close()
            conn.close()
            return True, f"Database: {db} | User: {user} | MySQL Version: {version}"
        else:
            return False, "Connection failed (is_connected returned False)"
    except MySQLError as e:
        return False, f"MySQL Error: {e}"
    except Exception as e:
        return False, f"Error: {e}"

# Test connection
success, message = test_database_connection(DB_CONFIG)

if success:
    print(f"✅ MySQL Connection Successful")
    print(f"   {message}")
    DB_CONNECTED = True
else:
    print(f"❌ MySQL Connection Failed")
    print(f"   {message}")
    DB_CONNECTED = False

In [ ]:
# 1.5 Verify Database Schema
if DB_CONNECTED:
    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor()
        
        # Check all 11 tables exist
        cursor.execute("""
            SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES 
            WHERE TABLE_SCHEMA = %s AND TABLE_TYPE = 'BASE TABLE'
            ORDER BY TABLE_NAME
        """, (DB_CONFIG['database'],))
        
        tables = [row[0] for row in cursor.fetchall()]
        
        expected_tables = [
            'announcements', 'announcement_history', 'announcement_keywords',
            'backup_logs', 'business_logs', 'buyers', 'keywords',
            'notifications', 'qualification_scores', 'sources', 'technical_logs'
        ]
        
        print("📊 Database Schema Verification:")
        print(f"\n  Found {len(tables)} tables:")
        for table in sorted(tables):
            status = "✅" if table in expected_tables else "❓"
            print(f"    {status} {table}")
        
        # Check row counts
        print("\n📈 Current Data:")
        for table in ['sources', 'keywords', 'buyers', 'announcements']:
            cursor.execute(f"SELECT COUNT(*) FROM {table}")
            count = cursor.fetchone()[0]
            print(f"    {table}: {count} rows")
        
        cursor.close()
        conn.close()
        
    except Exception as e:
        print(f"❌ Error verifying schema: {e}")

In [ ]:
# 1.6 Create data directories
data_dir = Path('../data')
data_dir.mkdir(exist_ok=True)

print(f"✅ Data directory ready: {data_dir}")
print(f"\n✨ Tab 1 (Setup) Complete!")
print(f"\nReady to proceed to Tab 2 (Extraction)")

---

## Tab 2: Data Extraction from APIs

In [ ]:
# 2.1 Extract from data.gouv.fr API

def extract_from_data_gouv_fr(keywords: List[str], limit: int = 100) -> pd.DataFrame:
    """
    Extract public market data from data.gouv.fr API.
    
    API Documentation: https://www.data.gouv.fr/api/1/
    Dataset: French public procurement announcements
    
    Args:
        keywords: List of keywords to search for
        limit: Max records to fetch
        
    Returns:
        DataFrame with raw announcement data
    """
    
    logger.info(f"Starting extraction from data.gouv.fr (keywords: {keywords})")
    
    # data.gouv.fr public datasets for markets
    # Using BOAMP (Bulletin Officiel des Annonces des Marchés Publics) dataset
    base_url = "https://www.data.gouv.fr/api/1/datasets/"
    
    # Search for BOAMP dataset
    search_url = "https://www.data.gouv.fr/api/1/datasets/?q=BOAMP+marchés+publics&page_size=1"
    
    try:
        print(f"📡 Fetching from data.gouv.fr...")
        response = requests.get(search_url, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        
        print(f"✅ Response received: {response.status_code}")
        
        # Parse datasets
        datasets = data.get('data', [])
        
        if datasets:
            print(f"Found {len(datasets)} dataset(s)")
            dataset = datasets[0]
            print(f"  Dataset: {dataset.get('title', 'Unknown')}")
            
            # Get resources (files)
            resources = dataset.get('resources', [])
            print(f"  Resources: {len(resources)} file(s)")
            
            # Look for CSV files
            csv_files = [r for r in resources if r.get('format', '').upper() == 'CSV']
            
            if csv_files:
                print(f"  CSV files available: {len(csv_files)}")
                # Use the most recent CSV
                csv_url = csv_files[0].get('url')
                
                if csv_url:
                    print(f"\n📥 Downloading CSV from {csv_url[:60]}...")
                    csv_response = requests.get(csv_url, timeout=30)
                    csv_response.raise_for_status()
                    
                    # Save raw CSV
                    raw_file = data_dir / 'data_gouv_fr_raw.csv'
                    with open(raw_file, 'wb') as f:
                        f.write(csv_response.content)
                    
                    print(f"✅ CSV saved: {raw_file}")
                    
                    # Parse CSV
                    df = pd.read_csv(raw_file, encoding='utf-8')
                    
                    print(f"\n📊 Data Summary:")
                    print(f"  Rows: {len(df)}")
                    print(f"  Columns: {list(df.columns)}")
                    
                    # Filter by keywords
                    if keywords:
                        print(f"\n🔍 Filtering by keywords: {keywords}")
                        mask = pd.Series([False] * len(df))
                        for keyword in keywords:
                            # Search in text columns
                            for col in df.columns:
                                if df[col].dtype == 'object':
                                    mask |= df[col].str.lower().str.contains(keyword.lower(), na=False)
                        
                        df_filtered = df[mask]
                        print(f"  Filtered rows: {len(df_filtered)} / {len(df)}")
                        return df_filtered.head(limit) if len(df_filtered) > 0 else df.head(limit)
                    
                    return df.head(limit)
        
        print(f"⚠️ No datasets found")
        return pd.DataFrame()
        
    except requests.RequestException as e:
        logger.error(f"Request error: {e}")
        print(f"❌ Error fetching from data.gouv.fr: {e}")
        return pd.DataFrame()
    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        print(f"❌ Unexpected error: {e}")
        return pd.DataFrame()

# Execute extraction
print("="*60)
print("EXTRACTION 1: data.gouv.fr API")
print("="*60)

keywords_to_search = CONFIG.get('keywords', {}).get('primary', ['modulaire', 'préfabriqué'])
df_data_gouv = extract_from_data_gouv_fr(keywords_to_search, limit=100)

if len(df_data_gouv) > 0:
    print(f"\n✅ Extraction successful: {len(df_data_gouv)} records")
    print(f"\nFirst record:")
    print(df_data_gouv.iloc[0] if len(df_data_gouv) > 0 else "No data")
else:
    print(f"\n⚠️ No data extracted from data.gouv.fr")

In [ ]:
# 2.2 Extract from BOAMP (Bulletin Officiel)

def extract_from_boamp(keywords: List[str], limit: int = 100) -> pd.DataFrame:
    """
    Extract public market data from BOAMP.
    
    BOAMP: Official bulletin for public market announcements
    Website: https://www.boamp.fr/
    
    Args:
        keywords: List of keywords to search for
        limit: Max records to fetch
        
    Returns:
        DataFrame with announcement data
    """
    
    logger.info(f"Starting extraction from BOAMP (keywords: {keywords})")
    
    # BOAMP API endpoint (if available)
    # Otherwise use web scraping or data.gouv.fr BOAMP dataset
    
    boamp_url = "https://www.data.gouv.fr/api/1/datasets/?q=bulletin+officiel+annonces+marchés&page_size=5"
    
    try:
        print(f"📡 Fetching BOAMP data from data.gouv.fr...")
        response = requests.get(boamp_url, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        datasets = data.get('data', [])
        
        all_data = []
        
        for dataset in datasets[:3]:  # Check top 3 datasets
            print(f"\n  Checking: {dataset.get('title', 'Unknown')[:50]}...")
            
            resources = dataset.get('resources', [])
            csv_files = [r for r in resources if r.get('format', '').upper() == 'CSV']
            
            if csv_files:
                csv_url = csv_files[0].get('url')
                
                if csv_url:
                    try:
                        print(f"    📥 Downloading CSV...")
                        csv_response = requests.get(csv_url, timeout=30)
                        csv_response.raise_for_status()
                        
                        # Try to parse CSV
                        from io import StringIO
                        df_temp = pd.read_csv(StringIO(csv_response.text), encoding='utf-8')
                        print(f"    ✅ Loaded {len(df_temp)} rows")
                        all_data.append(df_temp)
                        
                        if len(pd.concat(all_data, ignore_index=True)) >= limit:
                            break
                            
                    except Exception as e:
                        print(f"    ⚠️ Could not parse CSV: {str(e)[:50]}")
                        continue
        
        if all_data:
            df = pd.concat(all_data, ignore_index=True)
            
            # Save raw data
            raw_file = data_dir / 'boamp_raw.csv'
            df.to_csv(raw_file, index=False, encoding='utf-8')
            print(f"\n✅ BOAMP data saved: {raw_file}")
            
            return df.head(limit)
        
        print(f"\n⚠️ No BOAMP datasets found")
        return pd.DataFrame()
        
    except requests.RequestException as e:
        logger.error(f"Request error: {e}")
        print(f"❌ Error fetching BOAMP data: {e}")
        return pd.DataFrame()
    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        print(f"❌ Unexpected error: {e}")
        return pd.DataFrame()

print("\n" + "="*60)
print("EXTRACTION 2: BOAMP (Official Bulletin)")
print("="*60)

df_boamp = extract_from_boamp(keywords_to_search, limit=100)

if len(df_boamp) > 0:
    print(f"\n✅ BOAMP extraction successful: {len(df_boamp)} records")
else:
    print(f"\n⚠️ No data extracted from BOAMP")

In [ ]:
# 2.3 Combine all extracted data

print("\n" + "="*60)
print("DATA CONSOLIDATION")
print("="*60)

# Combine all data sources
dfs_to_combine = []

if len(df_data_gouv) > 0:
    df_data_gouv['source'] = 'data.gouv.fr'
    dfs_to_combine.append(df_data_gouv)
    print(f"\n✅ data.gouv.fr: {len(df_data_gouv)} records")

if len(df_boamp) > 0:
    df_boamp['source'] = 'BOAMP'
    dfs_to_combine.append(df_boamp)
    print(f"✅ BOAMP: {len(df_boamp)} records")

if dfs_to_combine:
    df_raw = pd.concat(dfs_to_combine, ignore_index=True)
    print(f"\n📊 Combined Dataset:")
    print(f"  Total rows: {len(df_raw)}")
    print(f"  Columns: {len(df_raw.columns)}")
    print(f"  Data size: {df_raw.memory_usage(deep=True).sum() / 1024:.2f} KB")
    
    # Save combined raw data
    raw_combined_file = data_dir / 'annonces_raw.csv'
    df_raw.to_csv(raw_combined_file, index=False, encoding='utf-8')
    print(f"\n✅ Raw data saved: {raw_combined_file}")
else:
    print(f"\n⚠️ No data extracted from any source")
    df_raw = pd.DataFrame()

print(f"\n✨ Tab 2 (Extraction) Complete!")
print(f"\nExtracted {len(df_raw)} total records")
print(f"Ready to proceed to Tab 3 (Transformation)")

---

## Summary

**Tab 1 (Setup)**: ✅ Configuration loaded, MySQL connection tested, schema verified

**Tab 2 (Extraction)**: ✅ Data extracted from data.gouv.fr and BOAMP APIs

**Next Steps**:
- Tab 3: Data Transformation (cleaning, normalization, region extraction, keyword extraction)
- Tab 4: Data Validation (doublon detection, date logic, amount validation)
- Tab 5: SQL Export (INSERT statements, CSV for bulk import)